# 00 — Set up mstr_robotics

Run this notebook top to bottom **once**, after installing the package and
deploying the Object Manager packages (see `docs/SETUP.md`).

It will:

1. create the folders the toolkit writes into
2. copy the config templates to their live filenames
3. *(you fill in your credentials)*
4. check the config for anything still left as a template
5. verify the connection to MicroStrategy
6. check that the configured object GUIDs exist in your environment

Every step prints `OK` or `FAIL` with the specific problem. Nothing is
overwritten and nothing is deleted.

## Step 1 — Create the output folders

`mstr_robotics._paths` computes where files go but does not create anything, and
`pandas.to_csv` will not create a missing parent directory. Without this step the
first export fails with *"Cannot save file into a non-existent directory"*.

Both `data/` and `output/` are gitignored, so nothing created here is committable.

In [ ]:
from mstr_robotics import setup

print(setup.ensure_dirs())

## Step 2 — Create your config files

Each `config/*.example.*` template is copied to its live filename. Files that
already exist are **left untouched**, so this is safe to re-run.

In [ ]:
print(setup.init_configs())

## Step 3 — Fill in your credentials  ⚠️ do this before continuing

Open **`config/user_d.yml`** and set:

| Key | Value |
|---|---|
| `base_url` | your MicroStrategy Library REST endpoint, ending in `/api` |
| `username` / `password` | a MicroStrategy account (a service account is preferable) |
| `project_id` | GUID of the project you want to work against |
| `pa_project_id` | GUID of your Platform Analytics project |

The templates ship with a reference environment's values — they will **not** work
against your server. Only fill in the other files if you need those features:
`mstr_redis_y.yml` (Redis), `dans_migrations.yml` (Azure migrations),
`API_KEY.env` (OpenAI / Perplexity).

Save the file, then run the next cell.

In [ ]:
print(setup.check_configs())

## Step 4 — Verify the connection

Confirms the credentials actually authenticate before anything else depends on them.

In [ ]:
print(setup.check_connection())

## Step 5 — Check the object GUIDs

`config/jupyter_objects_d.yml` holds the GUIDs of the objects the notebooks read.
This step **only checks that each GUID exists** — in the working project, or in
Platform Analytics for the usage reports and element prompts. Nothing is written
and no ID is guessed at.

Object *names* are not consulted: they drift from what the config records, so the
GUID is the only thing that counts.

An ID reported as missing means the Object Manager package providing it has not
been deployed, or the object was recreated — look the object up in MicroStrategy
and paste its GUID into `config/jupyter_objects_d.yml`.

In [ ]:
conn = setup.open_connection()

print(setup.check_object_ids(conn))

conn.close()

## Step 6 — Publish the cubes shipped with the packages

The Object Manager packages deploy the cubes the toolkit reads from, but a cube
arrives in your environment **unpublished** — it has a definition and no data.
Anything reading it (`jup_load_rag_cubes.ipynb`, the MCP servers) comes back empty
until the Intelligence Server has published it once.

This step collects every cube underneath `ele_cbe_folder_id` — subfolders
included — and publishes each one. Publishing is asynchronous, so each cube is
polled until the server reports it `READY`; large cubes can take a few minutes.
Re-running is safe, a published cube is simply refreshed.

In [ ]:
import yaml

from mstr_robotics._paths import CONFIG_DIR
from mstr_robotics.mstr_classes import MstrGlobal
from mstr_robotics.report import Cube

i_mstr_global = MstrGlobal()
i_cube = Cube()

RAG_Process = "turtorial_RAG"

nb_d = yaml.safe_load((CONFIG_DIR / "jupyter_objects_d.yml").read_text(encoding="utf-8"))
ele_cbe_folder_id = nb_d[RAG_Process]["ele_cbe_folder_id"]

conn = setup.open_connection(project_id=nb_d[RAG_Process]["project_id"])

# every cube stored underneath the folder, subfolders included
cube_list_l = i_mstr_global.get_cube_obj_l(conn=conn, folder_id=ele_cbe_folder_id)
print(f"{len(cube_list_l)} cube(s) found underneath {ele_cbe_folder_id}")
for cube_d in cube_list_l:
    print(f"  {cube_d['id']}  {cube_d['name']}")

pub_l = i_cube.publish_cubes(conn=conn, cube_l=cube_list_l)

conn.close()

## Done

If every step above says `OK`, your environment is ready.

Start with **`jup_prj_obj_exporter.ipynb`** to read objects out of a project, or
**`jup_schema_monitor.ipynb`** to monitor schema changes. `docs/SETUP.md` covers
the Object Manager packages and troubleshooting.